# Traffic Table Audit

## Objective

Perform a data quality assessment of the Traffic dataset before loading it into the Silver Layer.

The audit focuses on:

- Missing values
- Duplicate records
- Primary key validation
- Data type validation
- Category standardization
- Business rule validation

In [1]:
import pandas as pd

## 1. Load Dataset

Load the Traffic dataset into a Pandas DataFrame and perform an initial inspection.

In [2]:
traffic = pd.read_csv(r"C:\Dataset\traffic.csv")

## 2. Dataset Structure Review

Review:

- Number of records
- Number of columns
- Data types
- Overall dataset structure

In [3]:
traffic.info

<bound method DataFrame.info of         traffic_id  hour day_of_week    zone_type traffic_level  \
0                1     5         Thu        urban        medium   
1                2    18           6        urban           NaN   
2                3     5         Tue        urban           LOW   
3                4    10           2  residential          high   
4                5    23           3        Urban           low   
...            ...   ...         ...          ...           ...   
149995      149996     3           2      highway            lw   
149996      149997    10           6  RESIDENTIAL          high   
149997      149998    18         Wed        urban          high   
149998      149999    17           4        urban           NaN   
149999      150000    23           2        urban           low   

        avg_speed_kmph  
0            31.941595  
1            29.249695  
2            36.489258  
3            18.744747  
4            50.617655  
...          

In [4]:
traffic.dtypes

traffic_id          int64
hour                int64
day_of_week        object
zone_type          object
traffic_level      object
avg_speed_kmph    float64
dtype: object

## 3. Missing Value Analysis

Identify missing values across all columns and evaluate their impact on data completeness.

In [5]:
traffic.isnull().sum()

traffic_id            0
hour                  0
day_of_week        6000
zone_type          7500
traffic_level     10500
avg_speed_kmph     9000
dtype: int64

## 4. Duplicate Record Analysis

Check for fully duplicated records that may have resulted from ingestion or processing issues.

In [6]:
traffic.duplicated().sum()

np.int64(0)

## 5. Primary Key Validation

Evaluate whether `traffic_id` can serve as a reliable primary key by validating:

- NULL values
- Uniqueness
- Duplicate occurrences

In [7]:
traffic['traffic_id'].nunique()

145633

In [8]:
traffic['traffic_id'].duplicated().sum()

np.int64(4367)

## 6. Hour Column Validation

Validate hour values against the expected 24-hour clock range.

Expected range:

- 0 to 23

The objective is to identify:

- Negative values
- Values above 23
- Potential data quality issues

In [22]:
traffic['hour'].describe()

count    150000.000000
mean         11.567887
std           7.080935
min          -1.000000
25%           5.000000
50%          12.000000
75%          18.000000
max          25.000000
Name: hour, dtype: float64

In [29]:
traffic[traffic['hour'].between(0,23) ].count()

traffic_id        147000
hour              147000
day_of_week       141125
zone_type         139663
traffic_level     136703
avg_speed_kmph    138203
dtype: int64

In [30]:
traffic[~traffic['hour'].between(0,23) ].count()

traffic_id        3000
hour              3000
day_of_week       2875
zone_type         2837
traffic_level     2797
avg_speed_kmph    2797
dtype: int64

In [34]:
traffic[traffic['hour'] > 23 ].count()

traffic_id        2056
hour              2056
day_of_week       1971
zone_type         1959
traffic_level     1922
avg_speed_kmph    1926
dtype: int64

## 7. Day of Week Standardization Review

Review weekday values for:

- Multiple representations
- Missing values
- Standardization requirements

Examples:

- Monday
- Mon
- 1

In [28]:
traffic['day_of_week'].unique()

array(['Thu', '6', 'Tue', '2', '3', '4', '1', '0', 'Saturday', 'Thursday',
       nan, 'Sun', '5', 'Wednesday', 'Wed', 'Mon', 'Sunday', 'Fri',
       'Friday', 'Monday', 'Sat', 'Tuesday'], dtype=object)

## 8. Zone Type Standardization Review

Inspect zone categories for:

- Missing values
- Inconsistent casing
- Spelling variations

In [35]:
traffic['zone_type'].unique()

array(['urban', 'residential', 'Urban', nan, 'RESIDENTIAL', 'Highway',
       'urban ', 'highway', 'Residential'], dtype=object)

## 9. Traffic Level Standardization Review

Inspect traffic categories for:

- Missing values
- Inconsistent casing
- Spelling variations

In [12]:
traffic['traffic_level'].unique()

array(['medium', nan, 'LOW', 'high', 'low', 'low ', 'lw', 'Low', 'HIGH',
       'HIGH ', 'med ', 'meduim', 'High', 'Medium', 'med', 'hgh',
       'MEDIUM'], dtype=object)

## 10. Speed Validation

Validate average traffic speed measurements.

The objective is to identify:

- Missing values
- Negative speeds
- Unusual values
- Potential business rule violations

In [13]:
traffic[traffic['avg_speed_kmph'] < 0 ].count()

traffic_id        704
hour              704
day_of_week       675
zone_type         671
traffic_level     652
avg_speed_kmph    704
dtype: int64

In [14]:
traffic[traffic['avg_speed_kmph'] == 0 ].count()

traffic_id        1448
hour              1448
day_of_week       1399
zone_type         1366
traffic_level     1354
avg_speed_kmph    1448
dtype: int64

## 11. Audit Findings Summary

### Key Findings

- 6,000 missing day_of_week values
- 7,500 missing zone_type values
- 10,500 missing traffic_level values
- 9,000 missing avg_speed_kmph values
- 4,367 duplicate traffic_id values
- 3000 invalid hour values
- 704 negative speed values

### Candidate Primary Key Status

❌ traffic_id cannot currently be used as a primary key.

### Recommended Silver Layer Actions

- Resolve duplicate traffic_id values
- Handle missing values
- Standardize day_of_week values
- Standardize zone_type values
- Standardize traffic_level values
- Validate speed measurements